# PySpark Read & Write + File Formats

## CSV, JSON, Parquet, ORC, Save Modes

**Learning objective:** Understand file formats first, then practise 10
examples each for CSV, JSON, Parquet, ORC, and save modes. The examples
use PySpark DataFrames and illustrative paths. Adapt paths and
permissions for your environment.

# Part 1 --- File formats explained before operations

## 1. What is a file format?

A file format defines how values are represented and organized in a
file. It affects readability, parsing, schema handling, compression,
storage size, and query performance.

The same order data can be represented in different ways:

    order_id customer     amount status
  ---------- ---------- -------- -----------
        1001 Arun         250.50 Delivered
        1002 Meena         99.00 Pending

**CSV**

``` csv
order_id,customer,amount,status
1001,Arun,250.50,Delivered
1002,Meena,99.00,Pending
```

**JSON Lines**

``` json
{"order_id":1001,"customer":"Arun","amount":250.50,"status":"Delivered"}
{"order_id":1002,"customer":"Meena","amount":99.00,"status":"Pending"}
```

Parquet and ORC store the same logical records in binary columnar
layouts rather than readable comma-separated or JSON text.

A file format is not the same as a storage system: - **Format:** CSV,
JSON, Parquet, ORC. - **Storage:** local disk, HDFS, Amazon S3, Azure
Data Lake Storage, Google Cloud Storage. - **Compression codec:**
Snappy, gzip, zstd, zlib, and others depending on the format and
runtime.

For example, Parquet files can be stored on S3. Parquet describes the
representation; S3 describes where the bytes are stored.

## 2. Why format choice matters

1.  **Read performance:** How much data Spark needs to parse and scan.
2.  **Write performance:** Cost of serializing and storing records.
3.  **Storage size:** How well the format and compression reduce bytes.
4.  **Schema and types:** Whether types are stored or must be
    inferred/converted.
5.  **Interoperability:** Whether people and other systems can consume
    the output.
6.  **Column pruning:** Whether the reader can avoid reading unneeded
    columns.
7.  **Predicate pushdown:** Whether filters can be used to skip
    irrelevant data.
8.  **Cost:** Storage and compute costs for repeated analytical scans.

No format is best for every workload. CSV/JSON are common at system
boundaries; Parquet/ORC are common for analytics.

## 3. CSV in detail

**CSV means Comma-Separated Values.** It is plain text with fields
separated by a delimiter, usually a comma, and records commonly
separated by newlines.

Example:

``` csv
order_id,customer,amount,status
1001,Arun,250.50,Delivered
1002,Meena,99.00,Pending
```

With `header=True`, Spark treats the first line as column names. The
file stores text representations; it does not reliably encode business
types. For example, `00123` might be an identifier whose leading zeros
must be preserved, not an integer.

**Advantages** - Easy to inspect in a text editor. - Widely supported by
spreadsheets, databases, and applications. - Convenient for exchanging
simple tabular data.

**Limitations** - Type information is limited and must often be inferred
or declared. - Delimiters, quoting, embedded newlines, null markers, and
encoding need care. - Nested objects are awkward. - Parsing large
analytical scans can be less efficient than columnar storage.

A pipe-delimited file is still text:

``` text
order_id|customer|amount|status
1001|Arun|250.50|Delivered
```

Read it with `sep="|"`; the file extension does not determine its
delimiter.

**Best fit:** external tabular exports, simple interchange,
human-readable data, and systems that require CSV. It is common to
ingest CSV, validate and type-cast it, then write curated data to
Parquet or ORC.

## 4. JSON in detail

**JSON means JavaScript Object Notation.** It represents strings,
numbers, booleans, nulls, objects, and arrays. It is common in APIs,
logs, and event data.

A common Spark layout is JSON Lines / NDJSON: one complete JSON object
per line.

``` json
{"order_id":1001,"customer":"Arun","amount":250.50}
{"order_id":1002,"customer":"Meena","amount":99.00}
```

JSON can also represent nested objects and arrays:

``` json
{
  "order_id": 1001,
  "customer": {"customer_id": 501, "name": "Arun"},
  "items": [
    {"product": "Keyboard", "qty": 1},
    {"product": "Mouse", "qty": 2}
  ]
}
```

Spark can expose `customer` as a nested `struct` and `items` as an
`array` of structs. JSON is flexible, but fields can be missing or
change types between records. Schema inference is useful for
exploration, but production pipelines should validate schema and
required fields.

Spark's default JSON reader expects one self-contained record per line
(`multiLine=False`). For a conventional formatted multi-line JSON
document or array, use `multiLine=True`. Do not assume all JSON sources
have the same layout.

**Advantages:** human-readable, nested structures, natural fit for APIs
and semi-structured events. **Limitations:** parsing overhead, repeated
field names, variable schemas, and potentially larger files than
compressed columnar formats.

## 5. Parquet in detail

**Apache Parquet** is an open binary **columnar** format designed for
analytics. It stores schema/type metadata and organizes values in
structures such as row groups, column chunks, and pages, using encodings
and compression.

Conceptually, instead of treating a record as one text line, columns are
stored in column-oriented structures. If a report selects only
`order_id` and `amount` from a dataset with many columns, Spark can
often read only those columns. This is **column pruning**.

Parquet metadata and statistics may also help readers skip irrelevant
row groups for eligible filters. These optimizations depend on the file
layout, query, metadata, and reader support.

**Advantages** - Efficient for repeated analytical scans. - Typed schema
information. - Often compresses well. - Supports column pruning and
predicate pushdown when applicable. - Widely supported by Spark and
analytics tools.

**Limitations** - Not human-readable in a text editor. - Small writes
may create multiple part files. - Schema compatibility and evolution
still need governance. - It is not automatically fastest for every
workload.

**Best fit:** curated data-lake datasets, reporting, and analytical
processing. Parquet is a common general-purpose default for Spark
analytics.

## 6. ORC in detail

**ORC means Optimized Row Columnar.** Despite the word "Row," ORC is a
binary columnar format. It is common in Hadoop/Hive-oriented analytical
environments.

ORC files use structures such as **stripes**, column streams, metadata,
indexes/statistics, encodings, and compression. Readers may use these
statistics to avoid unnecessary work.

**Advantages** - Designed for analytical scans. - Stores schema and
typed values. - Supports compression, column projection, and
statistics-based skipping. - A natural fit in many Hive environments.

**Limitations** - Not human-readable as plain text. - Compatibility and
ecosystem conventions may make it less convenient in some
environments. - It is not automatically faster than Parquet; benchmark
real workloads.

**Best fit:** Hive-centric environments, existing ORC tables, or
workloads where testing shows ORC is a good fit.

## 7. Compare the four formats

  --------------------------------------------------------------------------------
  Feature          CSV            JSON             Parquet        ORC
  ---------------- -------------- ---------------- -------------- ----------------
  Representation   Plain text     Plain text       Binary         Binary

  Layout           Delimited      Objects/arrays   Columnar       Columnar
                   records                                        

  Human-readable   Yes            Yes              No             No

  Nested data      Awkward        Natural          Supported      Supported

  Stored           Limited        JSON value       Yes            Yes
  schema/types                    types; schema                   
                                  can vary                        

  Column pruning   Limited        Limited          Supported      Supported

  Common use       Interchange    APIs/events      Data-lake      Analytics/Hive
                                                   analytics      
  --------------------------------------------------------------------------------

This is not a universal speed ranking. Performance depends on data,
compression, partitions, storage, Spark settings, and queries.

## 8. What does read mean?

``` python
df = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
```

`spark.read` gives a `DataFrameReader`; `.csv(...)` selects the parser
and options; Spark identifies files and constructs a DataFrame. Schema
inference can require extra work. Transformations are generally lazy
until an action or write needs the results.

## 9. What does write mean?

``` python
df.write.mode("overwrite").parquet("output/orders_parquet")
```

`df.write` gives a `DataFrameWriter`; `mode(...)` defines what to do if
output exists; `.parquet(...)` selects the format and destination. Spark
commonly writes a **directory** containing one or more `part-...` files,
not one file with the directory's name. Exact output layout depends on
partitioning and the filesystem.

# Part 2 --- Common setup

Run this once before examples. Paths are illustrative and must exist/be
writable as appropriate.

``` python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, struct
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType
)

spark = SparkSession.builder.appName("ReadWriteFileFormats").getOrCreate()

data = [
    (1001, "Arun", 250.50, "Delivered", "India", "2026-10-01"),
    (1002, "Meena", 99.00, "Pending", "India", "2026-10-02"),
    (1003, "Ravi", 480.00, "Delivered", "Singapore", "2026-10-03"),
    (1004, "Divya", 125.75, "Cancelled", "India", "2026-10-04"),
    (1005, "John", 800.00, "Delivered", "UK", "2026-10-05")
]

schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("country", StringType(), True),
    StructField("order_date", StringType(), True)
])

orders_df = spark.createDataFrame(data, schema).withColumn(
    "order_date", to_date(col("order_date"), "yyyy-MM-dd")
)
orders_df.printSchema()
orders_df.show(truncate=False)
```

`StructType` defines the schema; `StructField` defines each column name,
type, and nullability. `to_date` converts date text into a Spark date
column. `DoubleType` is used for simplicity; financial systems often
prefer `DecimalType` for fixed precision and scale.

# Part 3 --- CSV: definition, options, and 10 examples

CSV is text with delimited fields. Spark parses it into typed DataFrame
columns.

## Important CSV parameters

-   `path`: input or output location.
-   `header`: whether the first row contains column names on read;
    whether to include column names on write.
-   `inferSchema`: infer types by inspecting data; useful for
    exploration but less predictable than a declared schema.
-   `schema`: explicit schema to apply when reading.
-   `sep` / `delimiter`: separator such as `","` or `"|"`.
-   `quote`: quoting character for fields containing separators or
    special characters.
-   `escape`: escape character inside quoted values.
-   `nullValue`: configured text marker for null values.
-   `encoding`: character encoding such as UTF-8, where supported.
-   `mode` on `spark.read.csv`: malformed-input handling. This is
    different from `df.write.mode(...)`, which is a save mode.

## CSV Example 1 --- Read with a header

``` python
csv_df = spark.read.csv("data/orders.csv", header=True)
csv_df.show(truncate=False)
csv_df.printSchema()
```

`header=True` treats the first row as column names. Because no schema or
inference is specified, columns are generally read as strings. `show`
displays values; `printSchema` shows types. Use this for a quick
inspection, then validate types.

## CSV Example 2 --- Infer types

``` python
csv_df = spark.read.csv(
    "data/orders.csv", header=True, inferSchema=True
)
```

`inferSchema=True` asks Spark to infer integer/numeric/date-like types
from input values. It can add work because Spark must inspect input
data, and inconsistent values may produce surprising results. Inspect
`printSchema()` and validate source quality.

## CSV Example 3 --- Apply an explicit schema

``` python
csv_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("country", StringType(), True),
    StructField("order_date", StringType(), True)
])
csv_df = spark.read.csv(
    "data/orders.csv", header=True, schema=csv_schema
)
```

`schema=csv_schema` supplies column names, order, and types rather than
relying on inference. `header=True` still consumes the header row.
Nullability describes the declared schema but does not replace
business-rule validation. Explicit schemas make production contracts
clearer.

## CSV Example 4 --- Pipe-delimited source

``` python
pipe_df = spark.read.csv(
    "data/orders_pipe.txt", header=True, sep="|", inferSchema=True
)
```

`sep="|"` tells Spark to split on pipes, not commas. The extension
`.txt` does not determine the delimiter. A wrong separator can lead to
malformed columns or a single column containing the whole line.

## CSV Example 5 --- Set character encoding

``` python
csv_df = spark.read.csv(
    "data/customers_utf8.csv", header=True, encoding="UTF-8"
)
```

`encoding` identifies how text bytes should be decoded. The setting must
match the actual source. It is useful for international names and
non-ASCII characters; choosing the wrong encoding can produce corrupted
text. Confirm support in the Spark version/environment.

## CSV Example 6 --- Configure a null marker

``` python
csv_df = spark.read.csv(
    "data/orders_with_nulls.csv",
    header=True, inferSchema=True, nullValue="NULL"
)
```

`nullValue="NULL"` configures the source token interpreted as null. The
source and consumer should agree on the convention. Do not choose a
marker that can also be a legitimate business value unless ambiguity is
handled.

## CSV Example 7 --- Write CSV with header

``` python
orders_df.write.csv("output/orders_csv", header=True)
```

`orders_df.write` creates a writer; `.csv(...)` selects CSV and
destination; `header=True` includes column names. Spark commonly creates
a directory containing one or more part files. If the path exists, the
default save mode generally fails.

## CSV Example 8 --- Write with a pipe delimiter

``` python
orders_df.write \
    .option("header", True) \
    .option("sep", "|") \
    .csv("output/orders_pipe")
```

`option("header", True)` enables the header; `option("sep", "|")`
chooses a pipe separator. `.csv(...)` writes the data. The receiving
system must expect the same delimiter.

## CSV Example 9 --- Write a null marker

``` python
orders_df.write \
    .option("header", True) \
    .option("nullValue", "NULL") \
    .csv("output/orders_null_marker")
```

The writer's `nullValue` option controls the configured textual
representation for null values. Use it only when the downstream consumer
expects that convention. Document the difference between a true null and
a literal string equal to the marker.

## CSV Example 10 --- Select columns and overwrite output

``` python
orders_df.select("order_id", "customer", "amount").write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/order_summary_csv")
```

`select(...)` returns a new DataFrame with only those fields, in that
order. `mode("overwrite")` replaces existing output at the destination
according to the filesystem's behavior. `header=True` writes column
names. The original `orders_df` is not mutated.

**CSV common mistakes:** wrong delimiter; forgotten header; assuming
values are typed; mishandling quotes/embedded delimiters; assuming a
write produces one file; using inference without validation; forgetting
save mode.

# Part 4 --- JSON: definition, options, and 10 examples

JSON represents objects, arrays, strings, numbers, booleans, and nulls.
Spark can map nested JSON objects to `struct` columns and arrays to
`array` columns.

`multiLine=False` is the common default and expects one complete JSON
record per line. Use `multiLine=True` for a conventional multi-line JSON
document/array. `schema` defines expected fields and types. Schema
inference is convenient for exploration but should not replace
validation.

## JSON Example 1 --- Read JSON Lines

``` python
json_df = spark.read.json("data/orders.json")
json_df.show(truncate=False)
json_df.printSchema()
```

The path can point to a JSON file or a directory of files. By default,
each line should contain one complete JSON record. Spark infers schema
if none is supplied. Inspect the schema, especially when records vary.

## JSON Example 2 --- Read with explicit schema

``` python
json_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("country", StringType(), True)
])
json_df = spark.read.schema(json_schema).json("data/orders.json")
```

`.schema(json_schema)` defines expected fields and types. Missing fields
can appear as null. Fields excluded from the schema may not appear in
the DataFrame. Validate malformed records and unexpected source fields
separately.

## JSON Example 3 --- Read a multi-line JSON document

``` python
json_df = spark.read.option("multiLine", True).json(
    "data/orders_array.json"
)
```

`multiLine=True` enables parsing of a document spread across multiple
lines, such as a formatted JSON array. It changes record framing; it
does not define the schema. Use it only when it matches the source
layout.

## JSON Example 4 --- Combine schema and multiline option

``` python
json_df = spark.read.schema(json_schema) \
    .option("multiLine", True) \
    .json("data/orders_formatted.json")
```

The schema defines expected columns/types; `multiLine=True` configures
how records are framed. These options solve separate problems. The file
must match both the structure and expected schema.

## JSON Example 5 --- Read a directory of JSON files

``` python
json_df = spark.read.json("data/incoming_orders/")
```

Spark reads files under the directory according to file-source behavior.
Keep unrelated files out of the source directory, because extra records
or fields may affect the result. Multiple files can introduce schema
variation.

## JSON Example 6 --- Read and select a nested field

Input record:

``` json
{"order_id":1001,"customer":{"id":501,"name":"Arun"},"items":[{"product":"Keyboard","qty":1}]}
```

``` python
nested_df = spark.read.json("data/nested_orders.json")
nested_df.printSchema()
nested_df.select("order_id", "customer.name").show(truncate=False)
```

Spark commonly represents `customer` as a nested struct and `items` as
an array of structs. `printSchema()` shows these nested types.
`"customer.name"` selects the nested `name` field. Validate nested
schemas as source applications evolve.

## JSON Example 7 --- Write JSON

``` python
orders_df.write.json("output/orders_json")
```

`.json(...)` serializes each row as JSON. Spark commonly creates
multiple part files and newline-delimited JSON records, rather than one
large JSON array. Existing output normally causes an error under the
default save mode.

## JSON Example 8 --- Overwrite JSON output

``` python
orders_df.write.mode("overwrite").json("output/orders_json")
```

`mode("overwrite")` requests replacement of existing output. This suits
a full-refresh job, but is destructive if the target path is wrong.
`.json(...)` selects the format.

## JSON Example 9 --- Write selected fields

``` python
orders_df.select("order_id", "customer", "status").write \
    .mode("overwrite").json("output/order_status_json")
```

`select(...)` limits output to the three named fields. The JSON records
will not include omitted fields such as amount, country, or order_date.
Overwrite replaces existing destination data.

## JSON Example 10 --- Write nested JSON

``` python
nested_output_df = orders_df.select(
    "order_id",
    struct(
        col("customer").alias("name"),
        col("country").alias("country")
    ).alias("customer")
)
nested_output_df.write.mode("overwrite").json("output/nested_orders")
```

`struct(...)` groups columns into a nested struct; inner `alias` calls
set nested field names; outer `alias("customer")` names the struct
column. The writer serializes that struct as a nested JSON object. The
output schema differs from the original flat schema.

**JSON common mistakes:** wrong multiline setting; assuming stable
schema; ignoring nested types; assuming output is one JSON array;
relying on inference as a production contract; including unrelated files
in the input directory.

# Part 5 --- Parquet: definition and 10 examples

Parquet is a binary columnar format. It stores schema/type metadata and
organizes data into row groups, column chunks, and pages. Column pruning
can reduce reads when only some columns are selected.
Metadata/statistics may enable skipping irrelevant row groups for
eligible predicates. Actual benefits depend on file layout and query.

## Parquet Example 1 --- Read Parquet

``` python
parquet_df = spark.read.parquet("data/orders_parquet")
parquet_df.show(truncate=False)
parquet_df.printSchema()
```

`.parquet(...)` selects the reader. The path often points to a directory
of part files. Spark reads schema metadata from Parquet; `printSchema()`
verifies types.

## Parquet Example 2 --- Write Parquet

``` python
orders_df.write.parquet("output/orders_parquet")
```

`.parquet(...)` serializes the DataFrame in Parquet format. Spark
commonly creates multiple part files. The default save mode generally
fails if the target already exists.

## Parquet Example 3 --- Overwrite Parquet

``` python
orders_df.write.mode("overwrite").parquet("output/orders_parquet")
```

`mode("overwrite")` requests replacement of existing output. Use it for
intended full refreshes only; confirm the path and understand the
storage behavior.

## Parquet Example 4 --- Select only two columns

``` python
parquet_df = spark.read.parquet("data/orders_parquet")
summary_df = parquet_df.select("order_id", "amount")
summary_df.show()
```

`select(...)` projects the two required columns. Parquet's columnar
layout allows Spark to often avoid reading unrelated columns. This is
column pruning; actual gains depend on the query and layout.

## Parquet Example 5 --- Filter rows

``` python
parquet_df = spark.read.parquet("data/orders_parquet")
delivered_df = parquet_df.filter(col("status") == "Delivered")
```

`filter(...)` keeps delivered records. Predicate pushdown and metadata
may help skip data, but they are optimizations rather than guarantees
that only matching bytes are read.

## Parquet Example 6 --- Snappy compression

``` python
orders_df.write.option("compression", "snappy") \
    .mode("overwrite").parquet("output/orders_snappy")
```

`compression="snappy"` requests Snappy compression where supported. It
often balances speed and compression, but test using representative data
and readers.

## Parquet Example 7 --- Zstandard compression

``` python
orders_df.write.option("compression", "zstd") \
    .mode("overwrite").parquet("output/orders_zstd")
```

`zstd` requests Zstandard compression if the runtime supports it.
Compression ratio and CPU costs vary; verify compatibility with every
downstream reader.

## Parquet Example 8 --- Partition by country

``` python
orders_df.write.mode("overwrite").partitionBy("country") \
    .parquet("output/orders_by_country")
```

`partitionBy("country")` creates directory partitions such as
`country=India`. It is directory partitioning, distinct from in-memory
Spark partitions. Filtering by country may enable partition pruning.
High-cardinality partition columns can create too many small
files/directories.

## Parquet Example 9 --- Read and filter a partitioned dataset

``` python
country_df = spark.read.parquet("output/orders_by_country")
india_df = country_df.filter(col("country") == "India")
```

Spark reads from the dataset root and can discover partition columns
from directory names. Filtering on `country` may skip unrelated
directories through partition pruning. Verify the schema and actual
execution plan.

## Parquet Example 10 --- Convert CSV to curated Parquet

``` python
raw_df = spark.read.csv(
    "data/orders.csv", header=True, inferSchema=True
)
typed_df = raw_df.withColumn("order_id", col("order_id").cast("int")) \
    .withColumn("amount", col("amount").cast("double"))
typed_df.write.mode("overwrite").parquet("output/orders_curated")
```

The CSV reader ingests the source; `withColumn(...cast(...))` makes
intended types explicit; overwrite rebuilds the curated destination;
Parquet writes columnar output. Invalid casts can produce null or raise
errors depending on Spark version and ANSI settings, so validate
conversions. For financial amounts, consider DecimalType.

**Parquet common mistakes:** trying to read it as plain text; assuming
one output file; over-partitioning; assuming every filter uses pushdown;
ignoring schema compatibility; selecting compression without
benchmarking.

# Part 6 --- ORC: definition and 10 examples

ORC (Optimized Row Columnar) is a binary columnar format used in
analytical systems, especially Hive-oriented environments. It uses
stripes, column streams, metadata, statistics/indexes, encodings, and
compression. It is not automatically faster than Parquet; test real
queries.

## ORC Example 1 --- Read ORC

``` python
orc_df = spark.read.orc("data/orders_orc")
orc_df.show(truncate=False)
orc_df.printSchema()
```

`.orc(...)` selects the ORC reader. The path can be a directory of ORC
part files. `printSchema()` verifies the exposed schema.

## ORC Example 2 --- Write ORC

``` python
orders_df.write.orc("output/orders_orc")
```

`.orc(...)` writes the DataFrame in ORC format. Spark may create
multiple part files; the default save mode generally fails if the
destination exists.

## ORC Example 3 --- Overwrite ORC

``` python
orders_df.write.mode("overwrite").orc("output/orders_orc")
```

Overwrite requests replacement of existing output. It is appropriate for
a full refresh only when that replacement is intended. Check the path
carefully.

## ORC Example 4 --- Select selected columns

``` python
orc_df = spark.read.orc("data/orders_orc")
orc_summary_df = orc_df.select("order_id", "amount")
```

`select(...)` requests only those fields. Because ORC is columnar, the
reader can often avoid reading unrelated column streams.

## ORC Example 5 --- Filter amount

``` python
orc_df = spark.read.orc("data/orders_orc")
high_value_df = orc_df.filter(col("amount") > 300)
```

The filter keeps rows where amount is greater than 300. ORC
metadata/statistics may help skip irrelevant data; actual optimization
depends on the files and query.

## ORC Example 6 --- Zlib compression

``` python
orders_df.write.option("compression", "zlib") \
    .mode("overwrite").orc("output/orders_orc_zlib")
```

`compression="zlib"` requests Zlib compression where supported.
Compression trades CPU for storage savings. Verify codec support and
reader compatibility in your environment.

## ORC Example 7 --- Snappy compression

``` python
orders_df.write.option("compression", "snappy") \
    .mode("overwrite").orc("output/orders_orc_snappy")
```

The option requests Snappy compression; overwrite makes repeated
practice runs easier; `.orc(...)` selects ORC output. Benchmark codec
choices on realistic data.

## ORC Example 8 --- Partition by status

``` python
orders_df.write.mode("overwrite").partitionBy("status") \
    .orc("output/orders_by_status_orc")
```

`partitionBy("status")` creates status-based directory partitions.
Queries filtering on status may skip other directories. Avoid excessive
partitions that produce small files.

## ORC Example 9 --- Read/filter partitioned ORC

``` python
status_df = spark.read.orc("output/orders_by_status_orc")
pending_df = status_df.filter(col("status") == "Pending")
```

Spark reads from the root directory and can discover partition columns
from directory names. Filtering may enable partition pruning. Use
`printSchema()` to verify the partition column is present.

## ORC Example 10 --- Write a filtered analytical result

``` python
analytics_df = orders_df.filter(col("status") == "Delivered").select(
    "order_id", "customer", "amount", "country", "order_date"
)
analytics_df.write.mode("overwrite").option("compression", "snappy") \
    .orc("output/delivered_orders_orc")
```

The filter selects delivered orders; `select` narrows the output;
overwrite controls existing destination behavior; compression configures
the codec; `.orc(...)` serializes the result. This combines filtering,
projection, and columnar output.

**ORC common mistakes:** assuming the name means row-oriented; assuming
it always beats Parquet; using unsupported codecs; over-partitioning;
assuming one output file; ignoring save modes.

# Part 7 --- Save modes: definition and 10 examples

A **save mode** controls what Spark does when the output destination
already exists. It is separate from malformed-record modes used by some
readers.

  -----------------------------------------------------------------------
  Mode                    Meaning                 Common use
  ----------------------- ----------------------- -----------------------
  `error` /               Fail if destination     Avoid accidental
  `errorifexists`         exists                  replacement

  `overwrite`             Replace existing output Full refresh

  `append`                Add output data         Additional batches,
                                                  with compatible schema

  `ignore`                Skip writing if         Do not recreate an
                          destination exists      existing path
  -----------------------------------------------------------------------

These modes do not automatically provide deduplication, upsert,
unique-key constraints, or transaction guarantees. Exact behavior
depends on the data source, table provider, catalog, filesystem, and
platform.

## Save Mode Example 1 --- Default mode

``` python
orders_df.write.parquet("output/default_mode")
```

No mode is specified, so the default is generally `errorifexists`. Spark
writes if the destination does not exist and normally raises an error if
it already exists. This helps avoid accidental replacement.

## Save Mode Example 2 --- Explicit errorifexists

``` python
orders_df.write.mode("errorifexists").parquet("output/error_mode")
```

The explicit mode communicates that an existing destination should cause
failure. It refers to the destination existing, not to whether the
DataFrame contains rows. `error` is commonly accepted as an alias.

## Save Mode Example 3 --- Overwrite

``` python
orders_df.write.mode("overwrite").parquet("output/overwrite_mode")
```

Overwrite requests replacement of existing output. Use for full
refreshes only. A wrong destination can destroy previous output, and
filesystem/table behavior should be understood.

## Save Mode Example 4 --- Append a second batch

``` python
batch_1 = spark.createDataFrame(
    [(2001, "Anu", 150.0), (2002, "Bala", 220.0)],
    ["order_id", "customer", "amount"]
)
batch_2 = spark.createDataFrame(
    [(2003, "Charan", 310.0)],
    ["order_id", "customer", "amount"]
)
batch_1.write.mode("overwrite").parquet("output/order_batches")
batch_2.write.mode("append").parquet("output/order_batches")
```

The first write creates a fresh dataset; the second appends another
batch. Both schemas should be compatible. Append does not deduplicate:
repeated order IDs may result in duplicate records. Appending files is
not the same as an upsert.

## Save Mode Example 5 --- Ignore existing destination

``` python
orders_df.write.mode("ignore").parquet("output/orders_once")
```

If the path exists, Spark skips writing; otherwise it writes. This does
not validate the existing data. "Path exists" does not mean the existing
output is correct or complete.

## Save Mode Example 6 --- Overwrite CSV

``` python
orders_df.write.mode("overwrite").option("header", True) \
    .csv("output/orders_csv_refresh")
```

The save mode controls destination behavior; `header=True` controls CSV
headers; `.csv(...)` selects the format. Save mode is independent of
file format.

## Save Mode Example 7 --- Append JSON events

``` python
events_df = spark.createDataFrame(
    [(3001, "PaymentReceived"), (3002, "OrderPacked")],
    ["event_id", "event_type"]
)
events_df.write.mode("append").json("output/event_log")
```

The DataFrame contains a new batch; append adds JSON output at the
destination. This is appropriate only when the destination is a
compatible dataset and duplicate/order concerns are handled. Appending
files does not guarantee chronological order.

## Save Mode Example 8 --- Append Parquet with matching schema

``` python
first_df = spark.createDataFrame([(4001, "Asha", 100.0)],
                                 ["order_id", "customer", "amount"])
next_df = spark.createDataFrame([(4002, "Dev", 200.0)],
                                ["order_id", "customer", "amount"])
first_df.write.mode("overwrite").parquet("output/append_demo")
next_df.write.mode("append").parquet("output/append_demo")
```

Both DataFrames have the same names and compatible types. The first
creates output; the second adds files. Production jobs should validate
schema explicitly before append.

## Save Mode Example 9 --- Demonstrate schema mismatch risk

``` python
first_df = spark.createDataFrame([(5001, "India")], ["order_id", "country"])
different_df = spark.createDataFrame([("5002", "UK")], ["order_id", "country"])
first_df.write.mode("overwrite").parquet("output/schema_risk")
different_df.write.mode("append").parquet("output/schema_risk")
```

The first `order_id` is an integer; the second is a string. Depending on
the source and Spark version, append may be rejected or produce a
dataset with incompatible file schemas. Do not rely on this as a
guaranteed error demonstration. Normalize and validate types before
writing:

``` python
normalized_df = different_df.withColumn("order_id", col("order_id").cast("int"))
```

Validate the cast result: invalid strings may become null or raise
errors depending on Spark settings.

## Save Mode Example 10 --- Generic writer with mode

``` python
validated_df = orders_df.filter(col("order_id").isNotNull())
validated_df.write.format("parquet").mode("overwrite").save(
    "output/validated_orders"
)
```

The filter retains rows with a non-null ID; `format("parquet")` selects
the data source; overwrite defines destination behavior; `save(path)`
writes using the configured format. Production validation should include
uniqueness, ranges, accepted statuses, and rejected-row handling.

## Save-mode mental model

-   `errorifexists`: "Do not replace existing output unexpectedly."
-   `overwrite`: "This job is intended to rebuild the destination."
-   `append`: "This is another batch; handle duplicates and schema
    compatibility."
-   `ignore`: "If the destination exists, leave it alone."

A save mode alone does not provide merge/upsert, deduplication,
concurrency coordination, or transactional guarantees. Use a table
format/platform with documented support when those guarantees are
required.

# Part 8 --- `format`, `option`, `options`, `load`, and `save`

## Example 1 --- Generic CSV reader

``` python
df = spark.read.format("csv").option("header", True) \
    .option("inferSchema", True).load("data/orders.csv")
```

`format("csv")` selects the source; each `option` configures a reader
setting; `load(path)` reads the source.

## Example 2 --- Generic JSON reader

``` python
df = spark.read.format("json").load("data/orders.json")
```

`format("json")` selects JSON parsing and `load` supplies the path. With
no schema, Spark infers one.

## Example 3 --- Generic Parquet writer

``` python
orders_df.write.format("parquet").mode("overwrite").save(
    "output/orders_generic"
)
```

`format` selects Parquet; `mode` defines existing-path behavior; `save`
performs the write.

## Example 4 --- Generic ORC writer with compression

``` python
orders_df.write.format("orc").option("compression", "snappy") \
    .mode("overwrite").save("output/orc_generic")
```

The format, compression, save mode, and path are configured separately.
Codec support depends on the runtime.

## Example 5 --- Set multiple CSV options

``` python
df = spark.read.format("csv").options(
    header=True, sep="|", inferSchema=True
).load("data/orders_pipe.txt")
```

`options(...)` supplies several reader options in one call. It is an
alternative to chaining individual `option(...)` calls.

# Part 9 --- End-to-end CSV-to-Parquet ETL

``` python
raw_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("data/orders.csv")

typed_df = raw_df \
    .withColumn("order_id", col("order_id").cast("int")) \
    .withColumn("amount", col("amount").cast("double")) \
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

valid_df = typed_df.filter(
    col("order_id").isNotNull() &
    col("amount").isNotNull() &
    col("order_date").isNotNull()
)

valid_df.write.format("parquet").mode("overwrite") \
    .partitionBy("country").save("output/curated_orders")
```

**Step by step** 1. Read the source CSV. 2. Use the header as column
names. 3. Infer types for this learning example; a production job often
benefits from a declared schema. 4. Cast ID and amount to intended
types. 5. Parse the date using `yyyy-MM-dd`. 6. Filter rows where
required conversions are not null. 7. Select Parquet as the curated
format. 8. Overwrite the prior curated result intentionally. 9.
Partition by country to support country-filtered queries, provided
cardinality and data volume justify it. 10. Save to the target path.

This is only a teaching example. Production pipelines should quarantine
rejected rows instead of silently dropping them, validate status/range
rules, track row counts, govern schema evolution, and select
partitioning based on query patterns.

# Part 10 --- Troubleshooting checklist

## Read failures

-   Confirm path, permissions, and storage connectors.
-   Confirm the correct format reader is used.
-   Check CSV header, delimiter, quoting, null marker, and encoding.
-   For JSON, confirm newline-delimited versus multi-line layout.
-   Inspect schema and malformed values.
-   Validate casts and required fields.

## Write failures

-   Check whether the destination already exists.
-   Select a save mode intentionally.
-   Confirm destination permissions and capacity.
-   Check schema and supported data types.
-   Confirm compression codec support.
-   Review Spark logs and avoid concurrent writes to the same path
    unless supported by the storage/table design.

## Too many small files

-   Inspect DataFrame partition count and output task count.
-   Consider `repartition` or `coalesce` only after considering data
    size and parallelism.
-   Avoid blindly using `coalesce(1)` for large datasets; it can
    bottleneck a distributed job.
-   Choose file and partition layout according to downstream query
    patterns.

# Part 11 --- Quick reference

``` python
# CSV
df = spark.read.csv("path", header=True, inferSchema=True)
df.write.mode("overwrite").option("header", True).csv("output/path")

# JSON Lines / NDJSON
df = spark.read.json("path")
df = spark.read.option("multiLine", True).json("multi_line_file")
df.write.mode("overwrite").json("output/path")

# Parquet
df = spark.read.parquet("path")
df.write.mode("overwrite").parquet("output/path")

# ORC
df = spark.read.orc("path")
df.write.mode("overwrite").orc("output/path")

# Generic writer
df.write.format("parquet").mode("overwrite").save("output/path")

# Partitioned output
df.write.mode("overwrite").partitionBy("country").parquet("output/path")

# Save modes
df.write.mode("errorifexists").parquet("output/path")
df.write.mode("overwrite").parquet("output/path")
df.write.mode("append").parquet("output/path")
df.write.mode("ignore").parquet("output/path")
```

## Final mental model

-   **CSV:** delimited text; good for simple interchange; types require
    care.
-   **JSON:** text objects/arrays; natural for nested/semi-structured
    sources.
-   **Parquet:** typed columnar storage; strong general-purpose
    analytics option.
-   **ORC:** typed columnar storage; common in Hive-oriented
    environments.
-   **Read:** interpret files as a DataFrame.
-   **Write:** serialize a DataFrame to a format and destination.
-   **Save mode:** decide what to do if the destination exists.
-   **Schema validation:** necessary even when the file format stores
    types.
-   **Partitioning:** can reduce scans but can also create many small
    files.
